# 06 – Data for the Tableau dashboard

The dashboard answers one question: if a cart reminder brings back X% of the abandoners, how many extra buyers do we get?
This notebook makes two small tables for it from the dbt table `mart_sessions` and saves them in `dashboard/data/`.

In [ ]:
from pathlib import Path
import pandas as pd

DATA_FILE = Path("../data/processed/mart_sessions_all.parquet")  # saved by notebook 04
OUT_DIR = Path("../dashboard/data")
OUT_DIR.mkdir(parents=True, exist_ok=True)

sessions = pd.read_parquet(DATA_FILE)
FULL_DATA_USERS = 12_899_779
print(sessions.shape)

## 1. Base numbers

One row with the numbers the dashboard needs. `full_data_factor` scales the 1M sample up to all 12.9M users.

In [ ]:
cart_users = sessions[sessions["buyer_type"].isin(["abandoner", "quick_buyer", "same_day_buyer", "slow_buyer"])]
abandoners = cart_users[cart_users["buyer_type"] == "abandoner"]

# target group: 3+ cart adds and 4+ active days (notebook 04)
target = abandoners[(abandoners["n_carts"] >= 3) & (abandoners["active_days"] >= 4)]

base = pd.DataFrame([{
    "abandoners": len(abandoners),
    "target_group": len(target),
    "current_buyers": sessions["has_order"].sum(),
    "cart_users": len(cart_users),
    "full_data_factor": round(FULL_DATA_USERS / len(sessions), 4),
}])
base.to_csv(OUT_DIR / "whatif_base.csv", index=False)
base

## 2. Extra buyers for every recovery rate

From 1% to 30% of the abandoners coming back after a reminder.

In [ ]:
factor = FULL_DATA_USERS / len(sessions)

curve = pd.DataFrame({"recovery_rate": [p / 100 for p in range(1, 31)]})
curve["extra_buyers"] = (len(abandoners) * curve["recovery_rate"]).round().astype(int)
curve["extra_buyers_full_data"] = (len(abandoners) * curve["recovery_rate"] * factor).round().astype(int)
curve.to_csv(OUT_DIR / "whatif_curve.csv", index=False)
curve.head()